# QUANT — Joint LOPO Validation Using Precomputed QUANT Features

This notebook performs a compact joint validation search over:

- `min_samples_leaf`
- Extra Trees class weighting
- probability threshold

It loads the already saved QUANT-transformed development matrix directly from:

```text
/content/drive/MyDrive/solar_flare_forecasting/Data/quant_transformer_features
```

The notebook **does not import, fit, or run `QUANTTransformer`**. If the precomputed feature file is missing, execution stops with a clear error instead of recomputing the transformation.

Leave-one-partition-out validation uses development Partitions **1, 2, 3, and 5**. Partition **4 remains reserved and is not loaded, trained on, ranked, or evaluated here**.

The notebook saves fold-level results, aggregated results, out-of-fold probabilities, experiment settings, and an interactive Lets-Plot TSS–HSS Pareto frontier. It does not choose a final configuration.


The notebook reads the five input-feature names and QUANT settings from `X_train_quant_5_features_metadata.json`; it does not require a separate `selected_feature_names.json` file.


## 1. Mount Google Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    "scikit-learn",
    "pandas",
    "tqdm",
    "pyarrow",
    "lets-plot",
]

IMPORT_NAMES = {
    "scikit-learn": "sklearn",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "pyarrow": "pyarrow",
    "lets-plot": "lets_plot",
}

missing_packages = []
for package_name, import_name in IMPORT_NAMES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])
else:
    print("All required packages are available.")


## 2. Imports and experiment configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from tqdm.auto import tqdm

from lets_plot import (
    LetsPlot,
    aes,
    geom_path,
    geom_point,
    ggplot,
    ggsave,
    ggsize,
    ggtitle,
    labs,
    layer_tooltips,
    scale_shape_manual,
    theme_minimal,
)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

LetsPlot.setup_html()

print("scikit-learn version:", sklearn.__version__)


In [ ]:
# =============================
# Precomputed data and output paths
# =============================
DATA_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/Data/quant_transformer_features"
)

PRECOMPUTED_QUANT_FILENAME = "X_train_quant_5_features.npy"
QUANT_METADATA_FILENAME = "X_train_quant_5_features_metadata.json"

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/"
    "Results/quant_joint_hyperparameter_validation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OOF_CHECKPOINT_DIR = CHECKPOINT_DIR / "oof_jobs"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OOF_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# =============================
# Dataset structure
# =============================
FEATURES_TO_USE = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]
DEVELOPMENT_PARTITIONS = [1, 2, 3, 5]
RESERVED_TEST_PARTITION = 4

# =============================
# Joint search grid
# =============================
MIN_SAMPLES_LEAF_OPTIONS = [1, 5, 10]
CLASS_WEIGHT_OPTIONS = [
    ("none", None),
    ("positive_5x", {0: 1, 1: 5}),
]
THRESHOLDS = np.arange(0.05, 1.00, 0.05)

# =============================
# QUANT settings used to create the saved feature matrix
# These are recorded and checked for reproducibility only.
# The transformation is not rerun.
# =============================
QUANT_INTERVAL_DEPTH = 6
QUANT_QUANTILE_DIVISOR = 4

# =============================
# Extra Trees and runtime settings
# =============================
N_ESTIMATORS = 200
RANDOM_SEED = 42
PREDICT_BATCH_SIZE = 20_000
FINITE_CHECK_BATCH_SIZE = 50_000

# Save one compressed Parquet file for each fitted LOPO job during validation,
# then combine those files into the requested all-OOF Parquet at the end.
PARQUET_COMPRESSION = "zstd"
WRITE_COMBINED_OOF_PARQUET = True

EXPECTED_CLASSIFIER_FITS = (
    len(MIN_SAMPLES_LEAF_OPTIONS)
    * len(CLASS_WEIGHT_OPTIONS)
    * len(DEVELOPMENT_PARTITIONS)
)
EXPECTED_COMPLETE_CONFIGURATIONS = (
    len(MIN_SAMPLES_LEAF_OPTIONS)
    * len(CLASS_WEIGHT_OPTIONS)
    * len(THRESHOLDS)
)

print("DATA_DIR:", DATA_DIR)
print("Precomputed QUANT file:", PRECOMPUTED_QUANT_FILENAME)
print("QUANT metadata file:", QUANT_METADATA_FILENAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Development partitions:", DEVELOPMENT_PARTITIONS)
print("Reserved test partition:", RESERVED_TEST_PARTITION)
print("Features:", FEATURES_TO_USE)
print("min_samples_leaf options:", MIN_SAMPLES_LEAF_OPTIONS)
print("Class weights:", [name for name, _ in CLASS_WEIGHT_OPTIONS])
print("Threshold count:", len(THRESHOLDS))
print("Expected fitted classifiers:", EXPECTED_CLASSIFIER_FITS)
print("Expected aggregated configurations:", EXPECTED_COMPLETE_CONFIGURATIONS)


## 3. Load the precomputed QUANT development features and sidecars

Only the transformed development matrix and its aligned development labels/partition IDs are loaded. No raw time-series tensor and no Partition 4 data are loaded.


In [ ]:
def require_file(data_dir: Path, filename: str) -> Path:
    path = data_dir / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "This notebook will not rerun QUANTTransformer. Confirm that the saved "
            "precomputed feature files are present in DATA_DIR."
        )
    return path


# Partition 4 files are deliberately not referenced or loaded.
x_train_quant_path = require_file(DATA_DIR, PRECOMPUTED_QUANT_FILENAME)
y_train_path = require_file(DATA_DIR, "y_train.npy")
train_partition_path = require_file(DATA_DIR, "train_partition_id.npy")
quant_metadata_path = require_file(DATA_DIR, QUANT_METADATA_FILENAME)

paths = {
    "X_train_quant": x_train_quant_path,
    "y_train": y_train_path,
    "train_partition_id": train_partition_path,
    "QUANT metadata": quant_metadata_path,
}

for name, path in paths.items():
    print(f"{name:24s} {path}")


In [ ]:
# The QUANT transformation is NOT fitted or executed in this notebook.
X_train_quant = np.load(x_train_quant_path, mmap_mode="r")
y_train = np.load(y_train_path)
train_partition_id = np.load(train_partition_path)

with open(quant_metadata_path, "r") as f:
    quant_feature_metadata = json.load(f)

if "features" not in quant_feature_metadata:
    raise KeyError(
        f"The metadata file {quant_metadata_path.name} does not contain a 'features' field."
    )

selected_feature_names = quant_feature_metadata["features"]

print("Loaded precomputed development data")
print("X_train_quant:      ", X_train_quant.shape, X_train_quant.dtype)
print("y_train:            ", y_train.shape, y_train.dtype)
print("train_partition_id: ", train_partition_id.shape, train_partition_id.dtype)
print("Feature names:      ", selected_feature_names)
print("QUANT metadata:     ", quant_metadata_path.name)
print("QUANT transformation executed here: False")


## 4. Validate alignment, features, partitions, and transformed values


In [ ]:
if X_train_quant.ndim != 2:
    raise ValueError(
        "Expected a two-dimensional precomputed QUANT feature matrix. "
        f"Received shape {X_train_quant.shape}."
    )

if X_train_quant.shape[0] != len(y_train):
    raise ValueError(
        f"Alignment error: {X_train_quant.shape[0]} transformed rows but "
        f"{len(y_train)} labels."
    )

if X_train_quant.shape[0] != len(train_partition_id):
    raise ValueError(
        f"Alignment error: {X_train_quant.shape[0]} transformed rows but "
        f"{len(train_partition_id)} partition IDs."
    )

if selected_feature_names != FEATURES_TO_USE:
    raise ValueError(
        f"{QUANT_METADATA_FILENAME} does not match the expected five-feature order.\n"
        f"Expected: {FEATURES_TO_USE}\n"
        f"Found:    {selected_feature_names}"
    )

metadata_n_cases = quant_feature_metadata.get("n_cases")
if metadata_n_cases is not None and int(metadata_n_cases) != len(y_train):
    raise ValueError(
        "The saved QUANT metadata reports a different number of development cases. "
        f"Metadata: {metadata_n_cases}; loaded labels: {len(y_train)}."
    )

metadata_interval_depth = quant_feature_metadata.get("interval_depth")
if (
    metadata_interval_depth is not None
    and int(metadata_interval_depth) != QUANT_INTERVAL_DEPTH
):
    raise ValueError(
        "The saved QUANT metadata uses a different interval_depth. "
        f"Expected {QUANT_INTERVAL_DEPTH}, found {metadata_interval_depth}."
    )

metadata_quantile_divisor = quant_feature_metadata.get("quantile_divisor")
if (
    metadata_quantile_divisor is not None
    and int(metadata_quantile_divisor) != QUANT_QUANTILE_DIVISOR
):
    raise ValueError(
        "The saved QUANT metadata uses a different quantile_divisor. "
        f"Expected {QUANT_QUANTILE_DIVISOR}, found {metadata_quantile_divisor}."
    )

observed_development_partitions = sorted(
    int(value) for value in np.unique(train_partition_id)
)
if observed_development_partitions != DEVELOPMENT_PARTITIONS:
    raise ValueError(
        "Unexpected development partitions. "
        f"Expected {DEVELOPMENT_PARTITIONS}, found {observed_development_partitions}."
    )

all_labels = np.unique(y_train)
if len(all_labels) != 2 or 1 not in all_labels:
    raise ValueError(
        "Expected binary labels containing positive class 1. "
        f"Found labels: {all_labels.tolist()}"
    )

# Verify the saved matrix without copying the full file into RAM.
for start in tqdm(
    range(0, len(X_train_quant), FINITE_CHECK_BATCH_SIZE),
    desc="Checking precomputed QUANT values",
):
    stop = min(start + FINITE_CHECK_BATCH_SIZE, len(X_train_quant))
    chunk = np.asarray(X_train_quant[start:stop])
    if not np.isfinite(chunk).all():
        bad_values = int((~np.isfinite(chunk)).sum())
        raise ValueError(
            f"Precomputed QUANT rows {start}:{stop} contain "
            f"{bad_values:,} NaN or infinite values."
        )

positive_label = 1
negative_label = int(all_labels[all_labels != positive_label][0])
train_original_indices = np.arange(len(y_train), dtype=np.int64)

print("Validated precomputed feature shape:", X_train_quant.shape)
print("Development partitions:", observed_development_partitions)
print("Negative label:", negative_label)
print("Positive label:", positive_label)
print("Partition 4 loaded: False")


In [ ]:
def class_count_table(y):
    values, counts = np.unique(y, return_counts=True)
    table = pd.DataFrame({"class": values, "count": counts})
    table["percent"] = 100 * table["count"] / len(y)
    return table


def partition_class_table(partition_ids, y):
    frame = pd.DataFrame({"partition": partition_ids, "label": y})
    counts = pd.crosstab(frame["partition"], frame["label"])
    counts.columns = [f"class_{column}" for column in counts.columns]
    counts["total"] = counts.sum(axis=1)
    for column in [c for c in counts.columns if c.startswith("class_")]:
        counts[f"{column}_percent"] = 100 * counts[column] / counts["total"]
    return counts.reset_index()


print("Development-set class counts")
display(class_count_table(y_train))
print("Development-set counts by partition")
display(partition_class_table(train_partition_id, y_train))


## 5. Confirm the precomputed QUANT feature source

The validation begins from `X_train_quant_5_features.npy`. There is no transformation fallback in this notebook: a missing or invalid precomputed matrix raises an error rather than rerunning QUANT.


In [ ]:
def index_hash(indices):
    return hashlib.sha256(
        np.asarray(indices, dtype=np.int64).tobytes()
    ).hexdigest()


def array_hash(values):
    return hashlib.sha256(np.asarray(values).tobytes()).hexdigest()


def file_signature(path: Path):
    stat = path.stat()
    return {
        "name": path.name,
        "size_bytes": int(stat.st_size),
        "modified_time_ns": int(stat.st_mtime_ns),
    }


PRECOMPUTED_QUANT_FEATURES_USED = True
QUANT_TRANSFORM_EXECUTED_IN_NOTEBOOK = False

print("Using precomputed QUANT features:", x_train_quant_path)
print("Transformed development shape:", X_train_quant.shape)
print("Memory-mapped dtype:", X_train_quant.dtype)
print("QUANTTransformer imported: False")
print("QUANT transformation executed: False")


## 6. Metrics, classifier, and checkpoint helpers

In [ ]:
def positive_class_scores(
    classifier,
    X,
    positive_label=1,
    batch_size=20_000,
    description="Predicting probabilities",
):
    classes = list(classifier.classes_)
    if positive_label not in classes:
        raise ValueError(f"Positive label {positive_label} not found in {classes}")
    positive_column = classes.index(positive_label)

    score_chunks = []
    for start in tqdm(
        range(0, len(X), batch_size),
        desc=description,
        leave=False,
    ):
        stop = min(start + batch_size, len(X))
        score_chunks.append(
            classifier.predict_proba(X[start:stop])[:, positive_column]
        )
    return np.concatenate(score_chunks)


def labels_from_threshold(scores, threshold, dtype=None):
    predictions = np.where(scores >= threshold, positive_label, negative_label)
    return predictions.astype(dtype) if dtype is not None else predictions


def binary_metrics(y_true, y_pred, scores=None):
    true_positive_mask = np.asarray(y_true) == positive_label
    pred_positive_mask = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        true_positive_mask,
        pred_positive_mask,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denominator = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (
        2 * ((tp * tn) - (fp * fn)) / hss_denominator
        if hss_denominator
        else np.nan
    )

    actual_positives = tp + fn
    predicted_positives = tp + fp
    frequency_bias = (
        predicted_positives / actual_positives
        if actual_positives
        else np.nan
    )

    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "recall_positive": recall_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "f1_positive": f1_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "pod_recall": pod,
        "false_positive_rate": fpr,
        "false_alarm_ratio": far,
        "tss": tss,
        "hss": hss,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "number_predicted_positive": int(predicted_positives),
        "frequency_bias": frequency_bias,
    }

    if scores is not None:
        output["roc_auc"] = roc_auc_score(true_positive_mask, scores)
        output["average_precision_pr_auc"] = average_precision_score(
            true_positive_mask, scores
        )

    return output


def build_extra_trees(class_weight, min_samples_leaf):
    return ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )

In [ ]:
FOLD_RESULTS_PATH = OUTPUT_DIR / "quant_lopo_fold_results.csv"
SUMMARY_RESULTS_PATH = OUTPUT_DIR / "quant_lopo_summary_results.csv"
PARETO_RESULTS_PATH = OUTPUT_DIR / "quant_lopo_pareto_results.csv"
VALIDATION_CONFIG_PATH = OUTPUT_DIR / "quant_lopo_validation_config.json"
COMBINED_OOF_PATH = OUTPUT_DIR / "quant_lopo_oof_probabilities.parquet"
PARETO_HTML_PATH = OUTPUT_DIR / "quant_lopo_pareto_interactive.html"


def class_weight_to_text(class_weight):
    return json.dumps(class_weight, sort_keys=True)


def model_job_id(min_samples_leaf, class_weight_name, held_out_partition):
    return (
        f"leaf_{int(min_samples_leaf)}"
        f"__weight_{class_weight_name}"
        f"__heldout_{int(held_out_partition)}"
    )


def complete_configuration_id(min_samples_leaf, class_weight_name, threshold):
    return (
        f"leaf_{int(min_samples_leaf)}"
        f"__weight_{class_weight_name}"
        f"__threshold_{float(threshold):.2f}"
    )


def atomic_write_csv(frame, path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp_path, index=False)
    os.replace(temp_path, path)
    try:
        os.sync()
    except AttributeError:
        pass


def atomic_write_parquet(frame, path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(
        temp_path,
        index=False,
        engine="pyarrow",
        compression=PARQUET_COMPRESSION,
    )
    os.replace(temp_path, path)
    try:
        os.sync()
    except AttributeError:
        pass


def atomic_write_json(payload, path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w") as f:
        json.dump(payload, f, indent=2)
    os.replace(temp_path, path)
    try:
        os.sync()
    except AttributeError:
        pass


def load_existing_fold_results():
    if not FOLD_RESULTS_PATH.exists():
        return pd.DataFrame()
    frame = pd.read_csv(FOLD_RESULTS_PATH)
    if "run_signature" not in frame.columns:
        raise ValueError(
            f"Existing fold results at {FOLD_RESULTS_PATH} do not contain a run_signature. "
            "Move or rename that file before starting this experiment."
        )
    return frame


def fold_result_job_is_complete(frame, job_id):
    if frame.empty:
        return False
    rows = frame[frame["job_id"] == job_id]
    if len(rows) != len(THRESHOLDS):
        return False
    observed = np.sort(rows["probability_threshold"].to_numpy(dtype=float))
    expected = np.sort(THRESHOLDS.astype(float))
    return np.allclose(observed, expected, rtol=0, atol=1e-12)


def oof_checkpoint_is_complete(path, expected_indices):
    if not path.exists():
        return False
    try:
        frame = pd.read_parquet(
            path,
            columns=["sample_index", "predicted_positive_probability"],
        )
    except Exception:
        return False

    if len(frame) != len(expected_indices):
        return False
    if frame["sample_index"].duplicated().any():
        return False
    if not np.isfinite(frame["predicted_positive_probability"]).all():
        return False
    return np.array_equal(
        np.sort(frame["sample_index"].to_numpy(dtype=np.int64)),
        np.sort(np.asarray(expected_indices, dtype=np.int64)),
    )


def replace_job_rows(existing_frame, new_rows_frame, job_id):
    if existing_frame.empty:
        combined = new_rows_frame.copy()
    else:
        combined = pd.concat(
            [existing_frame[existing_frame["job_id"] != job_id], new_rows_frame],
            ignore_index=True,
        )

    dedupe_columns = [
        "run_signature",
        "min_samples_leaf",
        "class_weight_name",
        "held_out_partition",
        "probability_threshold",
    ]
    combined = (
        combined
        .drop_duplicates(subset=dedupe_columns, keep="last")
        .sort_values(
            [
                "min_samples_leaf",
                "class_weight_name",
                "held_out_partition",
                "probability_threshold",
            ]
        )
        .reset_index(drop=True)
    )
    return combined

## 7. Save and verify the validation configuration

In [ ]:
def make_json_safe(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


signature_settings = {
    "development_partitions": DEVELOPMENT_PARTITIONS,
    "reserved_test_partition": RESERVED_TEST_PARTITION,
    "input_features": FEATURES_TO_USE,
    "precomputed_quant_feature_file": file_signature(x_train_quant_path),
    "quant_transform_executed_in_notebook": QUANT_TRANSFORM_EXECUTED_IN_NOTEBOOK,
    "quant_settings_used_to_create_saved_features": {
        "interval_depth": QUANT_INTERVAL_DEPTH,
        "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    },
    "extra_trees_fixed_settings": {
        "n_estimators": N_ESTIMATORS,
        "n_jobs": -1,
        "random_state": RANDOM_SEED,
    },
    "min_samples_leaf_values": MIN_SAMPLES_LEAF_OPTIONS,
    "class_weights": {
        name: class_weight for name, class_weight in CLASS_WEIGHT_OPTIONS
    },
    "threshold_values": THRESHOLDS.tolist(),
    "random_seed": RANDOM_SEED,
    "development_case_index_hash": index_hash(train_original_indices),
    "development_partition_hash": array_hash(train_partition_id),
    "development_label_hash": array_hash(y_train),
    "transformed_feature_shape": list(X_train_quant.shape),
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(make_json_safe(signature_settings), sort_keys=True).encode("utf-8")
).hexdigest()

if VALIDATION_CONFIG_PATH.exists():
    with open(VALIDATION_CONFIG_PATH, "r") as f:
        existing_config = json.load(f)
    existing_signature = existing_config.get("run_signature")
    if existing_signature != RUN_SIGNATURE:
        raise ValueError(
            "The output directory contains validation files from a different experiment.\n"
            f"Existing signature: {existing_signature}\n"
            f"Current signature:  {RUN_SIGNATURE}\n"
            "Use a different OUTPUT_DIR or archive the existing files before continuing."
        )
    run_started_utc = existing_config.get("run_started_utc")
    print("Matching validation configuration found. Resume mode is enabled.")
else:
    run_started_utc = datetime.now(timezone.utc).isoformat()

validation_config = {
    "run_signature": RUN_SIGNATURE,
    "run_started_utc": run_started_utc,
    "last_notebook_start_utc": datetime.now(timezone.utc).isoformat(),
    "validation_method": "leave-one-partition-out",
    "development_partitions": DEVELOPMENT_PARTITIONS,
    "reserved_test_partition": RESERVED_TEST_PARTITION,
    "partition_4_policy": "Reserved and not loaded or evaluated in this notebook.",
    "input_features": FEATURES_TO_USE,
    "data_dir": str(DATA_DIR),
    "precomputed_quant_feature_file": str(x_train_quant_path),
    "precomputed_quant_metadata_file": str(quant_metadata_path),
    "output_dir": str(OUTPUT_DIR),
    "precomputed_quant_features_used": PRECOMPUTED_QUANT_FEATURES_USED,
    "quant_transform_executed_in_notebook": QUANT_TRANSFORM_EXECUTED_IN_NOTEBOOK,
    "quant_settings_used_to_create_saved_features": {
        "interval_depth": QUANT_INTERVAL_DEPTH,
        "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    },
    "saved_quant_feature_metadata": quant_feature_metadata,
    "extra_trees_fixed_settings": {
        "n_estimators": N_ESTIMATORS,
        "n_jobs": -1,
        "random_state": RANDOM_SEED,
    },
    "min_samples_leaf_values": MIN_SAMPLES_LEAF_OPTIONS,
    "class_weights": {
        name: class_weight for name, class_weight in CLASS_WEIGHT_OPTIONS
    },
    "threshold_values": THRESHOLDS.tolist(),
    "random_seed": RANDOM_SEED,
    "expected_classifier_fits": EXPECTED_CLASSIFIER_FITS,
    "expected_complete_configurations": EXPECTED_COMPLETE_CONFIGURATIONS,
    "preprocessing": {
        "input_type": "precomputed QUANT feature matrix",
        "raw_time_series_loaded": False,
        "finite_values_verified": True,
        "development_cases": int(len(y_train)),
    },
    "source_files": {
        "X_train_quant": file_signature(x_train_quant_path),
        "y_train": file_signature(y_train_path),
        "train_partition_id": file_signature(train_partition_path),
        "quant_feature_metadata": file_signature(quant_metadata_path),
    },
    "transformed_feature_shape": list(X_train_quant.shape),
    "checkpointing": {
        "fold_results_path": str(FOLD_RESULTS_PATH),
        "oof_checkpoint_directory": str(OOF_CHECKPOINT_DIR),
        "parquet_compression": PARQUET_COMPRESSION,
    },
}

atomic_write_json(make_json_safe(validation_config), VALIDATION_CONFIG_PATH)
print("Run signature:", RUN_SIGNATURE)
print("Saved validation configuration:", VALIDATION_CONFIG_PATH)


## 8. Run the checkpointed joint LOPO validation

Each `(min_samples_leaf, class_weight, held-out partition)` job fits exactly one classifier. Its held-out probabilities are reused for the full threshold grid. Completed jobs are skipped only when both the fold-level threshold rows and the corresponding out-of-fold probability checkpoint are complete.

In [ ]:
fold_results_df = load_existing_fold_results()
if not fold_results_df.empty:
    unexpected_signatures = set(fold_results_df["run_signature"].dropna().unique()) - {
        RUN_SIGNATURE
    }
    if unexpected_signatures:
        raise ValueError(
            "Fold results contain rows from a different run signature: "
            f"{sorted(unexpected_signatures)}"
        )

jobs_skipped = 0
jobs_fitted = 0

for min_samples_leaf in MIN_SAMPLES_LEAF_OPTIONS:
    for class_weight_name, class_weight in CLASS_WEIGHT_OPTIONS:
        print("\n" + "=" * 88)
        print(
            f"min_samples_leaf={min_samples_leaf} | "
            f"class_weight={class_weight_name}: {class_weight}"
        )

        for held_out_partition in DEVELOPMENT_PARTITIONS:
            fold_train_idx = np.flatnonzero(
                train_partition_id != held_out_partition
            )
            fold_validation_idx = np.flatnonzero(
                train_partition_id == held_out_partition
            )

            y_fold_train = y_train[fold_train_idx]
            y_fold_validation = y_train[fold_validation_idx]

            if len(np.unique(y_fold_train)) != 2:
                raise ValueError(
                    f"Fold excluding partition {held_out_partition} does not contain both classes."
                )
            if len(np.unique(y_fold_validation)) != 2:
                raise ValueError(
                    f"Held-out partition {held_out_partition} does not contain both classes."
                )

            job_id = model_job_id(
                min_samples_leaf,
                class_weight_name,
                held_out_partition,
            )
            oof_job_path = OOF_CHECKPOINT_DIR / f"{job_id}.parquet"
            expected_sample_indices = train_original_indices[fold_validation_idx]

            fold_rows_complete = fold_result_job_is_complete(
                fold_results_df, job_id
            )
            oof_complete = oof_checkpoint_is_complete(
                oof_job_path, expected_sample_indices
            )

            if fold_rows_complete and oof_complete:
                jobs_skipped += 1
                print(f"Skipping completed job: {job_id}")
                continue

            print(
                f"Running {job_id}: "
                f"train={len(fold_train_idx):,}, "
                f"validation={len(fold_validation_idx):,}"
            )

            classifier = build_extra_trees(
                class_weight=class_weight,
                min_samples_leaf=min_samples_leaf,
            )

            fit_start = time.time()
            classifier.fit(X_train_quant[fold_train_idx], y_fold_train)
            fit_seconds = time.time() - fit_start

            validation_scores = positive_class_scores(
                classifier,
                X_train_quant[fold_validation_idx],
                positive_label=positive_label,
                batch_size=PREDICT_BATCH_SIZE,
                description=f"Partition {held_out_partition} probabilities",
            ).astype(np.float32)

            oof_job_df = pd.DataFrame({
                "run_signature": RUN_SIGNATURE,
                "job_id": job_id,
                "min_samples_leaf": int(min_samples_leaf),
                "class_weight_name": class_weight_name,
                "class_weight_value": class_weight_to_text(class_weight),
                "held_out_partition": int(held_out_partition),
                "sample_index": expected_sample_indices.astype(np.int64),
                "filtered_development_index": fold_validation_idx.astype(np.int64),
                "true_label": y_fold_validation,
                "predicted_positive_probability": validation_scores,
            })

            # Save original probabilities before thresholding.
            atomic_write_parquet(oof_job_df, oof_job_path)

            fold_rows = []
            for threshold in THRESHOLDS:
                validation_pred = labels_from_threshold(
                    validation_scores,
                    threshold,
                    dtype=y_fold_validation.dtype,
                )
                metrics = binary_metrics(
                    y_fold_validation,
                    validation_pred,
                    scores=validation_scores,
                )
                fold_rows.append({
                    "run_signature": RUN_SIGNATURE,
                    "job_id": job_id,
                    "configuration_id": complete_configuration_id(
                        min_samples_leaf,
                        class_weight_name,
                        threshold,
                    ),
                    "held_out_partition": int(held_out_partition),
                    "min_samples_leaf": int(min_samples_leaf),
                    "class_weight_name": class_weight_name,
                    "class_weight_value": class_weight_to_text(class_weight),
                    "probability_threshold": float(threshold),
                    "n_fold_train": int(len(fold_train_idx)),
                    "n_fold_validation": int(len(fold_validation_idx)),
                    "fold_train_positive": int(
                        (y_fold_train == positive_label).sum()
                    ),
                    "fold_validation_positive": int(
                        (y_fold_validation == positive_label).sum()
                    ),
                    "classifier_fit_seconds": float(fit_seconds),
                    **metrics,
                })

            new_job_rows_df = pd.DataFrame(fold_rows)
            fold_results_df = replace_job_rows(
                fold_results_df,
                new_job_rows_df,
                job_id,
            )
            atomic_write_csv(fold_results_df, FOLD_RESULTS_PATH)

            jobs_fitted += 1
            print(
                f"Completed {job_id} | fit time: {fit_seconds / 60:.2f} minutes | "
                f"saved {len(new_job_rows_df)} threshold rows"
            )

            del classifier, validation_scores, oof_job_df, new_job_rows_df
            gc.collect()

print("\nJoint LOPO validation checkpoint pass finished.")
print("Jobs fitted in this session:", jobs_fitted)
print("Jobs skipped as already complete:", jobs_skipped)
print("Current fold-level row count:", len(fold_results_df))
print("Fold results path:", FOLD_RESULTS_PATH)

## 9. Verify completion and combine the out-of-fold probabilities

In [ ]:
expected_fold_rows = EXPECTED_CLASSIFIER_FITS * len(THRESHOLDS)

fold_results_df = pd.read_csv(FOLD_RESULTS_PATH)
fold_results_df = fold_results_df[
    fold_results_df["run_signature"] == RUN_SIGNATURE
].copy()

if len(fold_results_df) != expected_fold_rows:
    raise RuntimeError(
        f"Expected {expected_fold_rows} fold-level rows after validation, "
        f"but found {len(fold_results_df)}. Re-run the validation cell to resume."
    )

expected_job_ids = []
for min_samples_leaf in MIN_SAMPLES_LEAF_OPTIONS:
    for class_weight_name, _ in CLASS_WEIGHT_OPTIONS:
        for held_out_partition in DEVELOPMENT_PARTITIONS:
            expected_job_ids.append(
                model_job_id(
                    min_samples_leaf,
                    class_weight_name,
                    held_out_partition,
                )
            )

missing_oof_files = [
    job_id
    for job_id in expected_job_ids
    if not (OOF_CHECKPOINT_DIR / f"{job_id}.parquet").exists()
]
if missing_oof_files:
    raise RuntimeError(
        "Missing out-of-fold probability checkpoints. Re-run the validation cell.\n"
        + "\n".join(missing_oof_files)
    )

if WRITE_COMBINED_OOF_PARQUET:
    oof_frames = [
        pd.read_parquet(OOF_CHECKPOINT_DIR / f"{job_id}.parquet")
        for job_id in expected_job_ids
    ]
    combined_oof_df = (
        pd.concat(oof_frames, ignore_index=True)
        .drop_duplicates(
            subset=[
                "run_signature",
                "min_samples_leaf",
                "class_weight_name",
                "held_out_partition",
                "sample_index",
            ],
            keep="last",
        )
        .sort_values(
            [
                "min_samples_leaf",
                "class_weight_name",
                "held_out_partition",
                "sample_index",
            ]
        )
        .reset_index(drop=True)
    )
    atomic_write_parquet(combined_oof_df, COMBINED_OOF_PATH)
    print("Saved combined OOF probabilities:", COMBINED_OOF_PATH)
    print("Combined OOF rows:", len(combined_oof_df))
    del oof_frames, combined_oof_df
    gc.collect()
else:
    print("Combined OOF writing is disabled. Per-job compressed files are available in:")
    print(OOF_CHECKPOINT_DIR)

print("Verified fold-level rows:", len(fold_results_df))
print("Verified OOF checkpoint files:", len(expected_job_ids))

## 10. Aggregate every complete configuration across the four LOPO folds

In [ ]:
# ============================================================
# SECTION 10 — Create a compact LOPO validation summary
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Prepare and validate the fold-level results
# ------------------------------------------------------------

fold_results_df = fold_results_df.copy()

CONFIGURATION_COLUMNS = [
    "min_samples_leaf",
    "class_weight_name",
    "probability_threshold",
]

CONFUSION_COLUMNS = [
    "true_negatives",
    "false_positives",
    "false_negatives",
    "true_positives",
]

REQUIRED_COLUMNS = (
    CONFIGURATION_COLUMNS
    + ["held_out_partition"]
    + CONFUSION_COLUMNS
)

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in fold_results_df.columns
]

if missing_columns:
    raise KeyError(
        "fold_results_df is missing required columns: "
        f"{missing_columns}"
    )


# Convert important columns to numeric explicitly
numeric_columns = [
    "min_samples_leaf",
    "probability_threshold",
    *CONFUSION_COLUMNS,
]

for column in numeric_columns:
    fold_results_df[column] = pd.to_numeric(
        fold_results_df[column],
        errors="raise",
    )

fold_results_df["class_weight_name"] = (
    fold_results_df["class_weight_name"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 2. Remove accidental duplicate checkpoint rows
# ------------------------------------------------------------

FOLD_KEY_COLUMNS = [
    "held_out_partition",
    "min_samples_leaf",
    "class_weight_name",
    "probability_threshold",
]

duplicate_count = int(
    fold_results_df.duplicated(
        subset=FOLD_KEY_COLUMNS,
        keep=False,
    ).sum()
)

if duplicate_count > 0:
    print(
        f"Warning: found {duplicate_count} duplicate fold rows. "
        "Keeping the most recently stored row for each result."
    )

    fold_results_df = (
        fold_results_df
        .drop_duplicates(
            subset=FOLD_KEY_COLUMNS,
            keep="last",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 3. Recalculate TSS and HSS from the confusion matrix
# ------------------------------------------------------------

tn = fold_results_df["true_negatives"].to_numpy(dtype=float)
fp = fold_results_df["false_positives"].to_numpy(dtype=float)
fn = fold_results_df["false_negatives"].to_numpy(dtype=float)
tp = fold_results_df["true_positives"].to_numpy(dtype=float)


# Positive-class recall / probability of detection
positive_recall = np.divide(
    tp,
    tp + fn,
    out=np.zeros_like(tp, dtype=float),
    where=(tp + fn) != 0,
)

# False-positive rate
false_positive_rate = np.divide(
    fp,
    fp + tn,
    out=np.zeros_like(fp, dtype=float),
    where=(fp + tn) != 0,
)

# True Skill Statistic
fold_results_df["tss"] = (
    positive_recall - false_positive_rate
)


# Heidke Skill Score
hss_numerator = 2.0 * ((tp * tn) - (fp * fn))

hss_denominator = (
    ((tp + fn) * (fn + tn))
    + ((tp + fp) * (fp + tn))
)

fold_results_df["hss"] = np.divide(
    hss_numerator,
    hss_denominator,
    out=np.zeros_like(hss_numerator, dtype=float),
    where=hss_denominator != 0,
)


# ------------------------------------------------------------
# 4. Ensure the other useful metrics are numeric
# ------------------------------------------------------------

USEFUL_METRICS = [
    "tss",
    "hss",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "false_alarm_ratio",
    "false_positive_rate",
    "frequency_bias",
]

missing_metrics = [
    metric
    for metric in USEFUL_METRICS
    if metric not in fold_results_df.columns
]

if missing_metrics:
    raise KeyError(
        "The following required metrics are missing from "
        f"fold_results_df: {missing_metrics}"
    )

for metric in USEFUL_METRICS:
    fold_results_df[metric] = pd.to_numeric(
        fold_results_df[metric],
        errors="coerce",
    )


# ------------------------------------------------------------
# 5. Aggregate useful metrics across the four LOPO folds
# ------------------------------------------------------------

metric_summary_df = (
    fold_results_df
    .groupby(
        CONFIGURATION_COLUMNS,
        dropna=False,
        sort=False,
    )[USEFUL_METRICS]
    .agg(["mean", "std"])
    .reset_index()
)


# Flatten the multi-level aggregate column names
flattened_columns = []

for column in metric_summary_df.columns:
    if isinstance(column, tuple):
        parts = [
            str(part)
            for part in column
            if str(part).strip()
        ]

        flattened_columns.append(
            "_".join(parts).lower()
        )
    else:
        flattened_columns.append(
            str(column).lower()
        )

metric_summary_df.columns = flattened_columns


# ------------------------------------------------------------
# 6. Keep mean confusion-matrix counts for interpretation
# ------------------------------------------------------------

confusion_summary_df = (
    fold_results_df
    .groupby(
        CONFIGURATION_COLUMNS,
        dropna=False,
    )[CONFUSION_COLUMNS]
    .mean()
    .reset_index()
    .rename(
        columns={
            "true_negatives": "true_negatives_mean",
            "false_positives": "false_positives_mean",
            "false_negatives": "false_negatives_mean",
            "true_positives": "true_positives_mean",
        }
    )
)


# ------------------------------------------------------------
# 7. Count completed folds per configuration
# ------------------------------------------------------------

fold_count_df = (
    fold_results_df
    .groupby(
        CONFIGURATION_COLUMNS,
        dropna=False,
    )["held_out_partition"]
    .nunique()
    .reset_index(name="n_completed_folds")
)


# ------------------------------------------------------------
# 8. Combine the compact summary
# ------------------------------------------------------------

summary_results_df = (
    metric_summary_df
    .merge(
        fold_count_df,
        on=CONFIGURATION_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    .merge(
        confusion_summary_df,
        on=CONFIGURATION_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)


# Add a readable unique configuration identifier
summary_results_df["configuration_id"] = (
    "leaf_"
    + summary_results_df["min_samples_leaf"]
      .astype(int)
      .astype(str)
    + "__weight_"
    + summary_results_df["class_weight_name"]
      .astype(str)
    + "__threshold_"
    + summary_results_df["probability_threshold"]
      .map(lambda value: f"{value:.2f}")
)


# ------------------------------------------------------------
# 9. Use a clean and useful column order
# ------------------------------------------------------------

FINAL_COLUMN_ORDER = [
    "configuration_id",
    "min_samples_leaf",
    "class_weight_name",
    "probability_threshold",
    "n_completed_folds",

    "tss_mean",
    "tss_std",
    "hss_mean",
    "hss_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "precision_positive_mean",
    "precision_positive_std",

    "recall_positive_mean",
    "recall_positive_std",

    "f1_positive_mean",
    "f1_positive_std",

    "false_alarm_ratio_mean",
    "false_alarm_ratio_std",

    "false_positive_rate_mean",
    "false_positive_rate_std",

    "frequency_bias_mean",
    "frequency_bias_std",

    "true_positives_mean",
    "false_positives_mean",
    "false_negatives_mean",
    "true_negatives_mean",
]

summary_results_df = summary_results_df[
    FINAL_COLUMN_ORDER
]


summary_results_df = (
    summary_results_df
    .sort_values(
        [
            "min_samples_leaf",
            "class_weight_name",
            "probability_threshold",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Validate completeness
# ------------------------------------------------------------

EXPECTED_COMPLETE_CONFIGURATIONS = (
    len(MIN_SAMPLES_LEAF_OPTIONS)
    * len(CLASS_WEIGHT_OPTIONS)
    * len(THRESHOLDS)
)

EXPECTED_FOLDS = len(DEVELOPMENT_PARTITIONS)

if len(summary_results_df) != EXPECTED_COMPLETE_CONFIGURATIONS:
    raise RuntimeError(
        f"Expected {EXPECTED_COMPLETE_CONFIGURATIONS} "
        f"configurations, but found {len(summary_results_df)}."
    )

incomplete_mask = (
    summary_results_df["n_completed_folds"]
    != EXPECTED_FOLDS
)

if incomplete_mask.any():
    display(
        summary_results_df.loc[
            incomplete_mask,
            [
                "configuration_id",
                "n_completed_folds",
            ],
        ]
    )

    raise RuntimeError(
        "At least one configuration is missing one or more "
        "LOPO folds."
    )


# ------------------------------------------------------------
# 11. Save the compact aggregated results
# ------------------------------------------------------------

output_directory = Path(OUTPUT_DIR)
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

summary_results_path = (
    output_directory
    / "quant_lopo_summary_results.csv"
)

# Use the notebook's atomic writer when available
if (
    "atomic_write_csv" in globals()
    and callable(atomic_write_csv)
):
    atomic_write_csv(
        summary_results_df,
        summary_results_path,
    )
else:
    summary_results_df.to_csv(
        summary_results_path,
        index=False,
    )


# ------------------------------------------------------------
# 12. Report and display
# ------------------------------------------------------------

print(
    "Successfully created compact summary for",
    len(summary_results_df),
    "configurations.",
)

print(
    "All configurations contain",
    EXPECTED_FOLDS,
    "LOPO folds.",
)

print(
    "Saved compact summary to:",
    summary_results_path,
)

print("\nSummary columns:")
print(summary_results_df.columns.tolist())

display(summary_results_df.head(20))

In [ ]:
# Save the aggregated summary results to Google Drive using the defined path
atomic_write_csv(summary_results_df, SUMMARY_RESULTS_PATH)

print(f"Summary results saved to: {SUMMARY_RESULTS_PATH}")

## 11. Calculate and save the full TSS–HSS Pareto table

In [ ]:
def pareto_optimal_mask(frame, tss_column="tss_mean", hss_column="hss_mean"):
    """Return True for rows not dominated on both full-precision TSS and HSS."""
    values = frame[[tss_column, hss_column]].to_numpy(dtype=float)
    valid = np.isfinite(values).all(axis=1)
    mask = np.zeros(len(frame), dtype=bool)

    for row_index, (tss_value, hss_value) in enumerate(values):
        if not valid[row_index]:
            continue

        dominates_row = (
            valid
            & (values[:, 0] >= tss_value)
            & (values[:, 1] >= hss_value)
            & (
                (values[:, 0] > tss_value)
                | (values[:, 1] > hss_value)
            )
        )
        dominates_row[row_index] = False
        mask[row_index] = not dominates_row.any()

    return mask


summary_results_df["is_pareto"] = pareto_optimal_mask(summary_results_df)

# Save every configuration; do not discard dominated points.
summary_results_df = summary_results_df.sort_values(
    ["min_samples_leaf", "class_weight_name", "probability_threshold"]
).reset_index(drop=True)

atomic_write_csv(summary_results_df, SUMMARY_RESULTS_PATH)
atomic_write_csv(summary_results_df, PARETO_RESULTS_PATH)

print("Saved aggregated results:", SUMMARY_RESULTS_PATH)
print("Saved full Pareto-marked results:", PARETO_RESULTS_PATH)
print("Total configurations:", len(summary_results_df))
print("Pareto-optimal configurations:", int(summary_results_df["is_pareto"].sum()))

## 12. Interactive Lets-Plot TSS–HSS Pareto graph

In [ ]:
plot_data = summary_results_df.copy()
plot_data["min_samples_leaf_label"] = (
    plot_data["min_samples_leaf"].astype(int).astype(str)
)
plot_data["pareto_status"] = np.where(
    plot_data["is_pareto"], "Pareto-optimal", "Dominated"
)

pareto_points = plot_data[plot_data["is_pareto"]].copy()
pareto_line = pareto_points.sort_values(
    ["tss_mean", "hss_mean", "min_samples_leaf", "probability_threshold"]
).copy()
pareto_line["frontier_group"] = "TSS-HSS Pareto frontier"

tooltip = (
    layer_tooltips()
    .line("min_samples_leaf|@min_samples_leaf")
    .line("Class weight|@class_weight_name")
    .line("Threshold|@probability_threshold")
    .line("Mean TSS|@tss_mean")
    .line("TSS std|@tss_std")
    .line("Mean HSS|@hss_mean")
    .line("HSS std|@hss_std")
    .line("Mean precision|@precision_positive_mean")
    .line("Mean recall|@recall_positive_mean")
    .line("Mean F1|@f1_positive_mean")
    .line("Mean balanced accuracy|@balanced_accuracy_mean")
    .line("Mean false-alarm ratio|@false_alarm_ratio_mean")
    .line("Mean false-positive rate|@false_positive_rate_mean")
    .line("Mean frequency bias|@frequency_bias_mean")
    .line("Pareto status|@pareto_status")
    .format("@probability_threshold", ".2f")
    .format("@tss_mean", ".4f")
    .format("@tss_std", ".4f")
    .format("@hss_mean", ".4f")
    .format("@hss_std", ".4f")
    .format("@precision_positive_mean", ".4f")
    .format("@recall_positive_mean", ".4f")
    .format("@f1_positive_mean", ".4f")
    .format("@balanced_accuracy_mean", ".4f")
    .format("@false_alarm_ratio_mean", ".4f")
    .format("@false_positive_rate_mean", ".4f")
    .format("@frequency_bias_mean", ".4f")
)

pareto_plot = (
    ggplot(plot_data, aes(x="tss_mean", y="hss_mean"))
    + geom_point(
        aes(
            color="class_weight_name",
            shape="min_samples_leaf_label",
        ),
        alpha=0.45,
        size=4.5,
        tooltips=tooltip,
    )
    # + geom_path(
    #     data=pareto_line,
    #     mapping=aes(
    #         x="tss_mean",
    #         y="hss_mean",
    #         group="frontier_group",
    #     ),
    #     inherit_aes=False,
    #     color="black",
    #     alpha=0.8,
    #     size=1.2,
    #     tooltips="none",
    # )
    + geom_point(
        data=pareto_points,
        mapping=aes(
            x="tss_mean",
            y="hss_mean",
            color="class_weight_name",
            shape="min_samples_leaf_label",
        ),
        inherit_aes=False,
        alpha=0.95,
        size=7.0,
        stroke=1.6,
        tooltips=tooltip,
    )
    + scale_shape_manual(values=[16, 17, 15])
    + labs(
        x="Mean validation TSS",
        y="Mean validation HSS",
        color="Class weight",
        shape="min_samples_leaf",
    )
    + ggtitle("QUANT LOPO Validation: TSS-HSS Pareto Frontier")
    + theme_minimal()
    + ggsize(1000, 700)
)

saved_html = ggsave(
    pareto_plot,
    PARETO_HTML_PATH.name,
    path=str(OUTPUT_DIR),
    iframe=False,
)
print("Saved interactive Pareto graph:", saved_html)

# Display the interactive graph directly in the notebook.
pareto_plot

## 13. Inspection tables — no automatic winner selection

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 220)

print("Complete aggregated configuration table")
display(summary_results_df)

print("Complete table sorted by mean TSS")
display(
    summary_results_df.sort_values(
        ["tss_mean", "hss_mean"],
        ascending=[False, False],
    ).reset_index(drop=True)
)

print("Complete table sorted by mean HSS")
display(
    summary_results_df.sort_values(
        ["hss_mean", "tss_mean"],
        ascending=[False, False],
    ).reset_index(drop=True)
)

print("Configurations marked as Pareto-optimal")
display(
    summary_results_df[summary_results_df["is_pareto"]]
    .sort_values(["tss_mean", "hss_mean"])
    .reset_index(drop=True)
)

## Final review step

Inspect the interactive Pareto plot and the saved validation tables before choosing the final combination of:

- `min_samples_leaf`
- `class_weight`
- `probability_threshold`

This notebook intentionally stops after validation. It does not select a winner, fit a final classifier, or evaluate the reserved test partition.